# 🚀 Setup Workshop — Data Engineering & Data Analytics

**Este notebook configura todo o ambiente necessário para o workshop.**

Execute todas as células em ordem. O script irá:
1. Carregar a configuração de catalog/schema (via `00_Config`)
2. Criar o catalog e schema (se necessário)
3. Importar os dados reais de tickets de TI a partir dos CSVs na pasta `data/`
4. Criar as tabelas Bronze (tb_tickets, tb_agents, tb_sla, tb_tickets_comments)
5. Criar a Skill de padronização para o Genie Code

**Pré-requisitos:**
- Permissão de CREATE CATALOG ou acesso a um catalog existente
- Compute Serverless ou cluster com acesso ao Unity Catalog

**Configuração:** Edite o notebook `00_Config` para definir catalog e schema do seu ambiente.

In [0]:
# ==========================================================
# Carrega configuração central (catalog, schema, DATA_DIR)
# Edite o notebook 00_Config para alterar os parâmetros
# ==========================================================

# MAGIC %run ./00_Config

In [0]:
# ==========================================================
# Criação do Catalog e Schema
# Se já existirem, não faz nada (IF NOT EXISTS)
# ==========================================================

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"✅ Catalog '{CATALOG}' e schema '{SCHEMA}' prontos")

In [0]:
import pandas as pd
from pyspark.sql.types import *

# ==========================================================
# Importação dos dados reais a partir dos CSVs
# Os CSVs estão na pasta data/ do workshop (versionados no git)
# ==========================================================

# Schemas explícitos para garantir tipos corretos
schemas = {
    "tb_tickets": StructType([
        StructField("ticket_id", LongType()),
        StructField("status", StringType()),
        StructField("priority", StringType()),
        StructField("source", StringType()),
        StructField("topic", StringType()),
        StructField("created_time", TimestampType()),
        StructField("close_time", TimestampType()),
        StructField("product_group", StringType()),
        StructField("support_level", StringType()),
        StructField("country", StringType()),
        StructField("latitude", DoubleType()),
        StructField("longitude", DoubleType())
    ]),
    "tb_agents": StructType([
        StructField("ticket_id", LongType()),
        StructField("agent_group", StringType()),
        StructField("agent_name", StringType()),
        StructField("agent_interactions", DoubleType())
    ]),
    "tb_sla": StructType([
        StructField("ticket_id", LongType()),
        StructField("expected_sla_to_resolve", TimestampType()),
        StructField("expected_sla_to_first_response", TimestampType()),
        StructField("first_response_time", TimestampType()),
        StructField("sla_for_first_response", StringType()),
        StructField("resolution_time", TimestampType()),
        StructField("sla_for_resolution", StringType()),
        StructField("survey_results", DoubleType())
    ]),
    "tb_tickets_comments": StructType([
        StructField("ticket_id", LongType()),
        StructField("comments", StringType())
    ])
}

for table_name, schema in schemas.items():
    csv_path = f"{DATA_DIR}/{table_name}.csv"
    df = spark.read.csv(csv_path, header=True, schema=schema)
    df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{table_name}")
    print(f"✅ {table_name}: {df.count()} registros importados de {table_name}.csv")

In [0]:
%sql
-- ==========================================================
-- Validação: Contagem das tabelas Bronze criadas
-- ==========================================================
SELECT 'tb_tickets' AS tabela, COUNT(*) AS registros FROM tb_tickets
UNION ALL
SELECT 'tb_agents', COUNT(*) FROM tb_agents
UNION ALL
SELECT 'tb_sla', COUNT(*) FROM tb_sla
UNION ALL
SELECT 'tb_tickets_comments', COUNT(*) FROM tb_tickets_comments;

In [0]:
# ==========================================================
# Metric View: mv_support_metrics
# Camada semântica centralizada para KPIs de suporte
# NOTA: Execute após rodar os notebooks ETL Silver
# ==========================================================

spark.sql(f"""
CREATE OR REPLACE METRIC VIEW {CATALOG}.{SCHEMA}.mv_support_metrics
AS SELECT

  -- Dimensões
  country                  AS `País`,
  priority                 AS `Prioridade`,
  status                   AS `Status`,
  topic                    AS `Tópico`,
  product_group            AS `Grupo de Produto`,
  support_level            AS `Nível de Suporte`,
  agent_group              AS `Grupo de Atendimento`,
  agent_name               AS `Atendente`,
  DATE_TRUNC('MONTH', created_time) AS `Mês de Abertura`,

  -- Medidas
  COUNT(ticket_id)         AS `Total de Tickets` COMMENT 'Quantidade total de tickets',
  ROUND(AVG(resolution_hours), 2)
                           AS `Tempo Médio de Resolução (h)` COMMENT 'MTTR em horas',
  ROUND(AVG(first_response_hours), 2)
                           AS `Tempo Médio de Primeira Resposta (h)` COMMENT 'Tempo até primeiro contato',
  ROUND(AVG(survey_results), 2)
                           AS `Nota Média de Satisfação` COMMENT 'CSAT (escala 1 a 5)',
  ROUND(SUM(CASE WHEN sla_resolution_met THEN 1 ELSE 0 END) / COUNT(ticket_id) * 100, 2)
                           AS `Taxa de SLA Resolução (%)` COMMENT '% tickets resolvidos no prazo',
  ROUND(SUM(CASE WHEN sla_first_response_met THEN 1 ELSE 0 END) / COUNT(ticket_id) * 100, 2)
                           AS `Taxa de SLA Primeira Resposta (%)` COMMENT '% primeira resposta no prazo',
  SUM(CASE WHEN status = 'Open' THEN 1 ELSE 0 END)
                           AS `Tickets Abertos` COMMENT 'Tickets ainda pendentes',
  SUM(CASE WHEN status = 'Closed' THEN 1 ELSE 0 END)
                           AS `Tickets Fechados` COMMENT 'Tickets já encerrados',
  ROUND(AVG(agent_interactions), 2)
                           AS `Média de Interações` COMMENT 'Interações médias por ticket'

FROM {CATALOG}.{SCHEMA}.silver_tickets_full
""")

print(f"✅ Metric View criada: {CATALOG}.{SCHEMA}.mv_support_metrics")

In [0]:
import os

# ==========================================================
# Cria skill de padronização para o Genie Code
# ==========================================================

skill_content = f"""---
name: workshop-etl-standards
description: Padrões de ETL e nomenclatura para o workshop de IT Support
---

# Padrões de ETL - Workshop IT Support

## Catalog e Schema
- **Catalog:** `{CATALOG}`
- **Schema:** `{SCHEMA}`

## Regras de Nomenclatura
- Tabelas Bronze: `tb_<entidade>` (ex: tb_tickets, tb_agents)
- Tabelas Silver: `silver_<entidade>_<sufixo>` (ex: silver_tickets_full)
- Tabelas Gold: `gold_<domínio>_<métrica>` (ex: gold_support_kpis)
- Tabelas SDP: `sdp_<camada>_<entidade>` (ex: sdp_silver_tickets_full)
- Tabelas Designer: `designer_<camada>_<entidade>` (ex: designer_silver_tickets)

## Padrões de Código SQL
- Sempre usar `INITCAP()` para padronizar colunas categóricas (priority, status, source)
- Sempre incluir `etl_timestamp` com `CURRENT_TIMESTAMP()` nas tabelas Silver e Gold
- Sempre validar `NOT NULL` em `ticket_id` e `country`
- Usar `LEFT JOIN` para não perder registros em joins
- Campos calculados devem ter nomes descritivos (ex: `resolution_hours`, `sla_resolution_met`)

## Tabelas Disponíveis (Bronze)
| Tabela | Descrição | Chave |
| --- | --- | --- |
| `tb_tickets` | Tickets de suporte | `ticket_id` |
| `tb_agents` | Atendentes | `ticket_id` |
| `tb_sla` | Métricas de SLA | `ticket_id` |
| `tb_tickets_comments` | Comentários | `ticket_id` |

## Padrões de Qualidade (SDP Expectations)
- Silver: usar `ON VIOLATION DROP ROW` para limpeza
- Gold crítico: usar `ON VIOLATION FAIL UPDATE`
- Gold monitoramento: usar sem cláusula (WARN)

## Idioma
- Comentários e documentação em português
- Nomes de colunas em inglês (snake_case)
- Metric View: nomes de dimensões e medidas em português
"""

# Salva a skill no diretório padrão do assistente
skill_dir = "/Workspace/Users/" + spark.sql("SELECT current_user()").collect()[0][0] + "/.assistant/skills/workshop-etl-standards"
os.makedirs(skill_dir, exist_ok=True)

with open(f"{skill_dir}/SKILL.md", "w") as f:
    f.write(skill_content)

print(f"✅ Skill criada em: {skill_dir}/SKILL.md")

## ✅ Setup Concluído!

### Próximos passos:

1. **Execute o notebook `ETL Tradicional/ETL Silver - IT Support`** para criar as tabelas Silver
2. **Execute o notebook `ETL Tradicional/ETL Gold - IT Support`** para criar as tabelas Gold
3. **Crie o pipeline SDP** usando o notebook em `SDP Pipeline/...` (menu + New → ETL Pipelines)
4. **Crie a Sala Genie** (use o Genie Code para ajudar! — veja seção 9.4 do roteiro)
5. **Crie o Dashboard AI/BI** (use o Genie Code! — veja seção 9.6 do roteiro)
6. **Crie o Job de orquestração** (ETL Silver → ETL Gold como tasks)
7. **Abra o roteiro** (`Roteiro Workshop DataEng e DataAnalysts`) e siga as seções

### Estrutura do Workshop:
```
Workshop - Plataforma End-to-End DataEng & DataAnalysts/
├── 00_Config                            ← Configuração central (catalog/schema)
├── 00_Setup_Workshop                    ← Este notebook (executar primeiro)
├── Roteiro Workshop DataEng e DataAnalysts  ← Guia da demo
├── ETL Tradicional/
│   ├── ETL Silver - IT Support          ← Notebook Silver (Seção 2)
│   └── ETL Gold - IT Support            ← Notebook Gold (Seção 2)
├── data/                                ← CSVs com dados reais (versionados no git)
│   ├── tb_tickets.csv
│   ├── tb_agents.csv
│   ├── tb_sla.csv
│   └── tb_tickets_comments.csv
└── SDP Pipeline/
    └── Pipeline SDP - IT Support/
        └── transformations/
            └── SDP ETL - IT Support (Silver e Gold)  ← Pipeline SDP (Seção 3)
```

> 💡 **Para replicar no ambiente do cliente:** edite o `00_Config` (altere `catalog` e `schema`), depois execute o `00_Setup_Workshop`. Todos os notebooks herdam a configuração automaticamente via `%run ./00_Config`.